# שיעור 24 — KNN על Penguins: מהכנת נתונים לתחזית ולהערכה

**הקשר:** תחנת מחקר רוצה להציע זיהוי מין פינגווין מתוך שתי מדידות. זהו תרגיל סיווג לימודי; אין כאן מערכת מאושרת לשימוש בשטח. המטרה היא לחזות species, לא לנבא מסה.

נגדיר מראש K=3, משקל שווה לשכנים ומרחק אוקלידי. K זה נבחר לצורך הלמידה, ואינו מוכרז כמיטבי. לא נשנה אותו על סמך Test.

עוגנים: [סיכום מפגש 5](../../07-KNN/Meeting-05-Didactic-Summary.md), [הכנת נתונים — מפגש 7](../../10-Data-Preparation/Meeting-07-Didactic-Summary.md). זוהי התאמה כיתתית חדשה של החומרים, ולא העתק של תרגיל מקור חסר.

## למה לומדים את זה עכשיו?
כדי לנחש סוג מוצר חדש צריך מאפיינים; את התשובה מסתירים מהמודל.
**בתהליך הפרויקט:** אימון וחיזוי.
**מסגרת פעילה:** 3 שעות לימוד — 135 דקות, כולל דוגמה ותרגול עצמאי. המורה בוחר את משימות הליבה שבמדריך; שאר הפעילויות לחזרה או לעיון, ללא תוספת שעות.

## לפני הקוד
בשיעור הקודם חישבנו מרחק והצבענו. כאן הספרייה עושה זאת עבורנו.
כתבו: מהי שורה? מהם שני מאפיינים? מה היעד? האם Id או species צריכים להיכלל ב־X?

הנתונים הם עותק Penguins במאגר הקורס: 344 רשומות, 7 עמודות. מקור המחקר: Kristen Gorman ו־Palmer Station LTER; תיעוד וייחוס: [palmerpenguins](https://github.com/allisonhorst/palmerpenguins). העותק אינו זהה לכל גרסה אחרת. נתונים אלה משמשים ללמידה; התאמה לפרויקט גמר דורשת בדיקה נפרדת.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

url = "https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df = pd.read_csv(url)
display(df.head())
display(df.isna().sum())

## 1. מפרידים קלט ויעד ומחלקים יחד
המאפיינים: אורך סנפיר במ״מ ומסה בגרמים. היעד טקסטואלי; KNN ב־sklearn מקבל תוויות טקסט ואין צורך ב־One-Hot ל־y.
שורה ללא יעד לא יכולה לשמש בדיקה מפוקחת. בעותק הנוכחי אין חסרים ב־species, ולכן הסינון אינו מסיר שורות.
stratify שומר בקירוב על יחסי המינים; random_state מאפשר שחזור, ואינו מבטיח חלוקה טובה מכל חלוקה אחרת.

In [ ]:
model_df = df.dropna(subset=["species"]).copy()
features = ["flipper_length_mm","body_mass_g"]
X = model_df[features].copy()
y = model_df["species"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print("Train:", len(X_train), "Test:", len(X_test))

## 2. חוזרים על הכנה מוכרת
קודם מחלקים, ואז מחשבים ממוצעי השלמה ומדדי תקנון **רק מאימון**. ל־Test מפעילים אותם פרמטרים.
לא כל מדידה מתחילה באותו קנה מידה: גרמים יכולים לשלוט במרחק על פני מ״מ. לאחר התקנון, כל מאפיין נמדד בסטיות תקן לפי האימון. זו בחירה למודל, לא הוכחת נורמליות.
עצירה: היכן מחושב מידע מהנתונים? מדוע אין כאן fit על Test?

In [ ]:
training_means = X_train.mean()
X_train_ready = X_train.fillna(training_means)
X_test_ready = X_test.fillna(training_means)

scaler = StandardScaler().set_output(transform="pandas")
scaler.fit(X_train_ready)
X_train_scaled = scaler.transform(X_train_ready)
X_test_scaled = scaler.transform(X_test_ready)
display(X_train_scaled.head())

## 3. fit לעומת predict
fit מקבל מדידות ותוויות אימון. ב־KNN הוא שומר את דוגמאות האימון הדרושות לחיפוש שכנים; הוא אינו לומד קו מפריד כמו רגרסיה.
predict מקבל מדידות בלבד ומחזיר תחזית לכל שורה באותו סדר. y_test אינו משתתף ביצירת התחזיות.
כל שכן מצביע קול אחד. המרחק מחושב בין המדידות המתוקננות.

In [ ]:
model = KNeighborsClassifier(n_neighbors=3, weights="uniform", metric="euclidean")
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)
print(predictions[:5])

## 4. השוואה ליעד שנשמר בצד
רק עכשיו משתמשים ב־y_test. האינדקס המקורי נשמר כדי לחבר כל תחזית לרשומה המתאימה; המערך predictions מסודר כמו X_test.

$$Accuracy=\frac{\text{מספר תחזיות נכונות}}{\text{מספר רשומות הבדיקה}}$$

לדוגמה מומצאת: 8 נכונות מתוך 10 נותנות 0.8, כלומר 80%. זה אינו הפלט הצפוי של Penguins.
לפני הרצה: הסבירו מה ההבדל בין תווית חזויה לבין ציון המודל.

In [ ]:
comparison = pd.DataFrame(
    {"actual": y_test, "predicted": predictions}, index=y_test.index
)
comparison["correct"] = comparison["actual"] == comparison["predicted"]
display(comparison.head(10))
correct_count = comparison["correct"].sum()
accuracy = accuracy_score(y_test, predictions)
print("Correct:", correct_count, "out of", len(y_test))
print("Accuracy:", accuracy, "Percent:", 100 * accuracy)

## 5. מבינים טעויות ומגבלות
ספרו טעויות וקראו עד שלוש דוגמאות. אם הטבלה ריקה, כתבו שלא נמצאו טעויות בחלוקה הזאת; אין לייצר שגיאות מלאכותיות.
Accuracy מסכם התאמה על חלוקה אחת. הוא אינו הסתברות שהתחזית המסוימת נכונה ואינו מבטיח ביצועים על אתר או אוכלוסייה אחרים.
איזה מין הוחלף באיזה מין? האם מחיר הטעות זהה לכל המינים? נלמד בהמשך מדדים לפי מחלקה.

In [ ]:
errors = comparison[comparison["correct"] == False]
print("Errors:", len(errors))
display(errors.head(10))

## כרטיס מודל לפרויקט
תעדו שאלה, יחידת תצפית, מקור, X ו־y, חלוקה, טיפול בחסרים, תקנון, K, מדד, דוגמת טעות ומגבלה.
אין להוסיף species למאפיינים, להשלים לפי כל המאגר, או לבחור K לפי Test. בחירת פרמטרים באימות בתוך האימון תילמד בהמשך, עם הכנה בכל קיפול.
**כרטיס יציאה:** הסבירו ללא קוד מה fit מקבל, מה predict מקבל, ומה צריך לדעת כדי לחשב Accuracy.